# HowTotrain.ipynb

This notebook demonstrates the YOLOv8 training pipeline step by step. It loads the custom `DetectionModel` from `model.py`, reads a single example image from the dataset (`.npz` files), runs a forward pass, computes IoU between a ground-truth bounding box and each predicted box to find the best match, defines a simple custom loss (`SimpleYOLOLoss` combining MSE for box regression and BCE for classification), and performs 2 epochs of training on that single example using the Adam optimizer.

<img src="training.jpg" width="1200" height="900" />

In [9]:
from model import DetectionModel
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

In [19]:
model = DetectionModel(nc=7)
model.eval()
print("")

In [20]:
df = pd.read_csv("../../Moon-Recognition/notebooks/all_boxes.csv")

In [21]:
example_path = df["npz_path"].unique().tolist()[0]
rows = df[df["npz_path"] == example_path]
data = np.load(example_path)
img = data['image']
#---------------------------------------------------
x = torch.from_numpy(img)
x = x.unsqueeze(0)

# IoU

**IoU (Intersection over Union)** measures the overlap between a predicted bounding box and a ground-truth box. It is the ratio of their intersection area to their union area:

$$
\text{IoU} = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{intersection}}{\text{area}(A) + \text{area}(B) - \text{intersection}}
$$

For axis-aligned boxes with corners $(x_1, y_1, x_2, y_2)$:
- $X_1 = \max(x_1, x_{p1}), \quad X_2 = \min(x_2, x_{p2})$
- $Y_1 = \max(y_1, y_{p1}), \quad Y_2 = \min(y_2, y_{p2})$
- $\text{intersection} = \max(0, X_2 - X_1) \times \max(0, Y_2 - Y_1)$

In YOLO, IoU is used during both **training** (as part of the loss function, e.g., CIOU) and **inference** (for NMS — Non-Maximum Suppression, where boxes with IoU > threshold are suppressed). The cell below computes IoU between the ground-truth box and each of the 1344 predictions to find the best-matching prediction.

In [22]:
path, classId, x_c, y_c, w,h = df.iloc[3]
x1 = x_c - w/2
x2 = x_c + w/2
y1 = y_c - h/2
y2 = y_c + h/2

In [23]:
IoU = 0 
for i in range(1344):
    xp_c, yp_c, wp, hp  = model(x)[0,:4,i]
    
    xp1 = xp_c - wp/2
    xp2 = xp_c + wp/2
    yp1 = yp_c - hp/2
    yp2 = yp_c + hp/2

    X1 = max(x1,xp1)
    X2 = min(x2,xp2)
    Y1 = max(y1,yp1)
    Y2 = min(y2,yp2)

    IoU1 = ((X2-X1)*(Y2-Y1))/((x2-x1)*(y2-y1) + (xp2-xp1)*(yp2-yp1) - (X2-X1)*(Y2-Y1))

    if IoU1>IoU:
        IoU = IoU1
        i_best = i
        
print(IoU)
print(i_best)

tensor(0.2535, grad_fn=<DivBackward0>)
136


In [24]:
import torch


class SimpleYOLOLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss() 
        self.mse = nn.MSELoss()             
        
    def forward(self, preds, targets):
        box_preds = preds[:4]    
        cls_preds = preds[4:]    

        loss_box = self.mse(box_preds, targets[:4])
        loss_cls = self.bce(cls_preds, targets[4:])
        return loss_box + loss_cls, loss_box.item(), loss_cls.item()

In [25]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = SimpleYOLOLoss()

for epoch in range(2):
    optimizer.zero_grad()          
    
    preds = model(x)[0,:,i_best]      
    
    fake_targets = torch.tensor([x_c,y_c,w,h ,1,0,0,0,0,0,0])
    
    loss, l_box, l_cls = criterion(preds, fake_targets) 
    
    loss.backward()                
    
    optimizer.step()               
    
    print(f"Epoch {epoch+1:2d} | Total: {loss:.4f} | Box: {l_box:.4f} | Cls: {l_cls:.4f}")

Epoch  1 | Total: 53.2977 | Box: 52.4039 | Cls: 0.8938
Epoch  2 | Total: 53.2167 | Box: 52.3232 | Cls: 0.8935
